# Day 3: Disney Parks Guest Spending Behavior

You are a data analyst working with the Disney Parks revenue team to understand nuanced guest spending patterns across different park experiences. The team wants to develop a comprehensive view of visitor purchasing behaviors. Your goal is to uncover meaningful insights that can drive personalized marketing strategies.

In [ ]:
import pandas as pd
import numpy as np

fct_guest_spending_data = [
  {
    "guest_id": 1,
    "visit_date": "2024-07-05",
    "amount_spent": 50,
    "park_experience_type": "Attraction"
  },
  {
    "guest_id": 2,
    "visit_date": "2024-07-06",
    "amount_spent": 30,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 3,
    "visit_date": "2024-07-10",
    "amount_spent": 20.5,
    "park_experience_type": "Retail"
  },
  {
    "guest_id": 4,
    "visit_date": "2024-07-12",
    "amount_spent": 40,
    "park_experience_type": "Entertainment"
  },
  {
    "guest_id": 1,
    "visit_date": "2024-07-15",
    "amount_spent": 35,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 5,
    "visit_date": "2024-07-20",
    "amount_spent": 60,
    "park_experience_type": "Attraction"
  },
  {
    "guest_id": 6,
    "visit_date": "2024-07-25",
    "amount_spent": 25,
    "park_experience_type": "Retail"
  },
  {
    "guest_id": 1,
    "visit_date": "2024-08-03",
    "amount_spent": 55,
    "park_experience_type": "Attraction"
  },
  {
    "guest_id": 1,
    "visit_date": "2024-08-15",
    "amount_spent": 45,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 2,
    "visit_date": "2024-08-05",
    "amount_spent": 22,
    "park_experience_type": "Retail"
  },
  {
    "guest_id": 2,
    "visit_date": "2024-08-20",
    "amount_spent": 38,
    "park_experience_type": "Entertainment"
  },
  {
    "guest_id": 7,
    "visit_date": "2024-08-10",
    "amount_spent": 15,
    "park_experience_type": "Character Meet"
  },
  {
    "guest_id": 3,
    "visit_date": "2024-08-25",
    "amount_spent": 28,
    "park_experience_type": "Retail"
  },
  {
    "guest_id": 3,
    "visit_date": "2024-08-27",
    "amount_spent": 32,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 1,
    "visit_date": "2024-09-02",
    "amount_spent": 65,
    "park_experience_type": "Attraction"
  },
  {
    "guest_id": 8,
    "visit_date": "2024-09-05",
    "amount_spent": 50,
    "park_experience_type": "Retail"
  },
  {
    "guest_id": 9,
    "visit_date": "2024-09-15",
    "amount_spent": 40,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 10,
    "visit_date": "2024-09-20",
    "amount_spent": 70,
    "park_experience_type": "Entertainment"
  },
  {
    "guest_id": 1,
    "visit_date": "2024-09-25",
    "amount_spent": 35,
    "park_experience_type": "Dining"
  },
  {
    "guest_id": 8,
    "visit_date": "2024-09-28",
    "amount_spent": 10,
    "park_experience_type": "Character Meet"
  }
]
fct_guest_spending = pd.DataFrame(fct_guest_spending_data)


## Question 1

What is the average spending per guest per visit for each park experience type during July 2024? Ensure that park experience types with no recorded transactions are shown with an average spending of 0.0. This analysis helps establish baseline spending differences essential for later segmentation.

In [ ]:
# Note: pandas and numpy are already imported as pd and np
# The following tables are loaded as pandas DataFrames with the same names: fct_guest_spending
# Please print your final result or dataframe

def park_experience(fct_guest_spending):
  fct_guest_spending['visit_date'] = pd.to_datetime(fct_guest_spending['visit_date'])
  
  july_data = fct_guest_spending[
    (fct_guest_spending['visit_date'].dt.year == 2024) &
    (fct_guest_spending['visit_date'].dt.month == 7)
  ]

  spending_per_guest = july_data.groupby(['guest_id', 'visit_date', 'park_experience_type'])['amount_spent'].sum().reset_index()

  avg_spending = spending_per_guest.groupby('park_experience_type')['amount_spent'].mean().reset_index()

  #all park types
  all_types = fct_guest_spending['park_experience_type'].unique()
  all_types_df = pd.DataFrame({'park_experience_type' : all_types})
  
  final_result = all_types_df.merge(avg_spending, on='park_experience_type', how='left')
  final_result['amount_spent'] = final_result['amount_spent'].fillna(0.0)

  final_result.rename(columns = {'amount_spent' : 'avg_spending_per_visit'}, inplace=True)

  print(final_result)

print(park_experience(fct_guest_spending))

## Question 2

For guests who visited our parks more than once in August 2024, what is the difference in spending between their first and their last visit? This investigation, using sequential analysis, will reveal any shifts in guest spending behavior over multiple visits.

In [ ]:
fct_guest_spending['visit_date'] = pd.to_datetime(fct_guest_spending['visit_date'])

# filter for August 2024
august_data = fct_guest_spending[
    (fct_guest_spending['visit_date'].dt.year == 2024) &
    (fct_guest_spending['visit_date'].dt.month == 8)
]

# sum amount spent per visit
visit_totals = august_data.groupby(['guest_id', 'visit_date'])['amount_spent'].sum().reset_index()

# identifying first and last visit
def get_spending_diff(group):
  if len(group) < 2:
    return None
  group_sorted = group.sort_values('visit_date')
  first = group_sorted.iloc[0]['amount_spent']
  last = group_sorted.iloc[-1]['amount_spent']

  return pd.Series({
    'first_visit_spent': first,
    'last_visit_spent': last,
    'spending_difference': last - first
  })

result = visit_totals.groupby('guest_id').apply(get_spending_diff).dropna().reset_index()

print(result.head())

## Question 3

In September 2024, how can guests be categorized into distinct spending segments such as Low, Medium, and High based on their total spending? Use the following thresholds for categorization: 
-Low: Includes values from $0 up to, but not including, $50.
-Medium: Includes values from $50 up to, but not including, $100.
-High: Includes values from $100 and above. 
Exclude guests who did not make any purchases in the period.

In [ ]:
fct_guest_spending['visit_date'] = pd.to_datetime(fct_guest_spending['visit_date'])

#filter by september 2024
sept_2024 = fct_guest_spending[
  (fct_guest_spending['visit_date'].dt.year == 2024) &
  (fct_guest_spending['visit_date'].dt.month == 9)
]

#group by guests with sum spending
total_spending = sept_2024.groupby('guest_id')['amount_spent'].sum().reset_index()

# removing guests with 0 spending
total_spending = total_spending[total_spending['amount_spent'] > 0]

def categorize_spending(amount):
  if amount < 50:
    return 'Low'
  elif amount < 100:
    return 'Medium'
  else:
    return 'High'

total_spending['spending_segment'] = total_spending['amount_spent'].apply(categorize_spending)
print(total_spending.head())

Made with ❤️ by [Interview Master](https://www.interviewmaster.ai)